# Curadoria — Pipeline FALSO (Google Fact Check)

Lê todos os CSVs raw do pipeline `pipeline_falso_google_factcheck/raw/`,
acumula o histórico, aplica limpeza e padronização e salva um CSV curated com timestamp.

**Regras desta camada:**
- Remover HTML do `texto_afirmacao`
- Normalizar espaços e capitalização
- Padronizar datas para `YYYY-MM-DD`
- Remover duplicatas por `texto_principal + fonte + url_origem`
- Mapear `avaliacao_original` para categoria normalizada
- Adicionar colunas do schema obrigatório curated
- **Não** modificar arquivos raw
- **Não** gerar dataset final de treino

**v2 — Correções nesta versão:**
- Mapa de avaliações expandido (+12 entradas)
- Normalização unicode (NFKC) antes do lookup
- Fallback ASCII para encoding corrompido (`não_é_bem_assim`)
- Fallback por prefixo para ratings em forma de sentença
- `confere` adicionado como VERDADEIRO


## Bibliotecas

In [1]:
import re
import unicodedata
import uuid
import pandas as pd

from datetime import datetime
from pathlib import Path
from html.parser import HTMLParser

## Configuração de caminhos

In [2]:
NOME_PIPELINE = "pipeline_falso_google_factcheck"

PASTA_RAW     = Path(f"../dados/{NOME_PIPELINE}/raw")
PASTA_CURATED = Path(f"../dados/{NOME_PIPELINE}/curated")
PASTA_CURATED.mkdir(parents=True, exist_ok=True)

print(f"Raw:     {PASTA_RAW}")
print(f"Curated: {PASTA_CURATED}")

Raw:     ..\dados\pipeline_falso_google_factcheck\raw
Curated: ..\dados\pipeline_falso_google_factcheck\curated


## Funções utilitárias

In [3]:
class _StripHTML(HTMLParser):
    """Parser simples que descarta tags e acumula apenas o texto."""
    def __init__(self):
        super().__init__()
        self._partes = []

    def handle_data(self, data):
        self._partes.append(data)

    def get_text(self):
        return " ".join(self._partes)


def remover_html(texto: str) -> str:
    """Remove tags HTML e normaliza espaços em branco."""
    if not isinstance(texto, str) or not texto.strip():
        return ""
    parser = _StripHTML()
    parser.feed(texto)
    limpo = parser.get_text()
    limpo = re.sub(r"\s+", " ", limpo).strip()
    return limpo


def padronizar_data(valor) -> str:
    """
    Tenta converter datas em vários formatos para YYYY-MM-DD.
    Retorna string vazia se não conseguir.
    """
    if not isinstance(valor, str) or not valor.strip():
        return ""
    valor = valor.strip()
    formatos = [
        "%Y-%m-%dT%H:%M:%SZ",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d",
        "%d/%m/%Y",
        "%d/%m/%Y %H:%M:%S",
    ]
    for fmt in formatos:
        try:
            return datetime.strptime(valor, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return ""


# ===========================================================================
# MAPA DE AVALIAÇÕES — v2 (corrigido e expandido)
# Expandido para cobrir valores antes classificados incorretamente como OUTRO.
# Chaves sempre em minúsculo — a função normaliza antes de consultar.
# ===========================================================================
_MAPA_AVALIACAO = {
    # --- FALSO ---
    "falso":                   "FALSO",
    "false":                   "FALSO",
    "incorreto":               "FALSO",
    "incorrect":               "FALSO",
    "mentira":                 "FALSO",
    "errado":                  "FALSO",           # novo: 107 ocorrências
    "insustentável":           "FALSO",           # novo: 37 ocorrências
    "insustentavel":           "FALSO",           # novo: variante sem acento
    "montagem":                "FALSO",           # novo: imagem manipulada (9x)
    "sátira":                  "FALSO",           # novo: usada para disseminar falso (7x)
    "satira":                  "FALSO",           # novo: variante sem acento
    "boato":                   "FALSO",           # novo: 1 ocorrência
    "predominantemente falso": "FALSO",           # novo: 1 ocorrência
    # --- ENGANOSO ---
    "enganoso":                "ENGANOSO",
    "misleading":              "ENGANOSO",
    "distorcido":              "ENGANOSO",
    "parcialmente falso":      "ENGANOSO",
    "mostly false":            "ENGANOSO",
    "half true":               "ENGANOSO",
    "não_é_bem_assim":         "ENGANOSO",        # novo: Boatos.org (91x)
    "nao_e_bem_assim":         "ENGANOSO",        # novo: fallback ASCII de encoding
    "não é bem assim":         "ENGANOSO",        # novo: variante com espaços
    "nao e bem assim":         "ENGANOSO",        # novo: variante ASCII com espaços
    "enganador":               "ENGANOSO",        # novo: 15-17 ocorrências
    # --- FORA_DE_CONTEXTO ---
    "fora de contexto":        "FORA_DE_CONTEXTO",
    "sem contexto":            "FORA_DE_CONTEXTO",
    "out of context":          "FORA_DE_CONTEXTO",
    "falta contexto":          "FORA_DE_CONTEXTO", # novo: 4 ocorrências
    # --- IMPRECISO ---
    "impreciso":               "IMPRECISO",
    "exagerado":               "IMPRECISO",
    # --- NAO_VERIFICAVEL ---
    "não verificável":         "NAO_VERIFICAVEL",
    "nao verificavel":         "NAO_VERIFICAVEL",  # variante sem acento
    "unverified":              "NAO_VERIFICAVEL",
    # --- VERDADEIRO ---
    "verdadeiro":              "VERDADEIRO",
    "true":                    "VERDADEIRO",
    "comprovado":              "VERDADEIRO",
    "certo":                   "VERDADEIRO",
    "correto":                 "VERDADEIRO",
    "fato":                    "VERDADEIRO",
    "fato verificado":         "VERDADEIRO",
    "confirmado":              "VERDADEIRO",
    "confere":                 "VERDADEIRO",       # novo: ausente no mapa anterior
}


def _normalizar_chave(texto: str) -> str:
    """
    Normaliza texto para lookup no mapa:
    1. NFKC — resolve compatibilidade unicode
    2. Remove U+FFFD gerado por encoding corrompido nos arquivos raw
    3. Colapsa underscores múltiplos (artefato da remoção de chars)
    4. Colapsa espaços múltiplos
    5. Strip + lower
    """
    if not isinstance(texto, str):
        return ""
    texto = unicodedata.normalize("NFKC", texto)
    texto = texto.replace("\ufffd", "")
    texto = re.sub(r"_+", "_", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip().lower()


_TRANSLITERACAO = str.maketrans(
    "ãáâàéêèíîóôõúûçÃÁÂÀÉÊÈÍÎÓÔÕÚÛÇ",
    "aaaaeeeiiooouucAAAAEEEIIOOOUUC",
)


def _ascii_simples(texto: str) -> str:
    """Transliteração PT→ASCII para fallback de encoding corrompido."""
    return texto.translate(_TRANSLITERACAO)


# Prefixos que identificam o veredicto mesmo em ratings de forma de sentença.
# Ex.: 'Falso: O vídeo mostra...' → prefixo 'falso:' → FALSO
_PREFIXOS_CATEGORIA = [
    ("falso:",                "FALSO"),
    ("falso -",               "FALSO"),
    ("é falso",               "FALSO"),
    ("e falso",               "FALSO"),
    ("sao falsas",            "FALSO"),
    ("sao falsos",            "FALSO"),
    ("enganoso:",             "ENGANOSO"),
    ("enganoso -",            "ENGANOSO"),
    ("é enganoso",            "ENGANOSO"),
    ("e enganoso",            "ENGANOSO"),
    ("enganosa",              "ENGANOSO"),
    ("verdadeiro:",           "VERDADEIRO"),
    ("verdadeiro -",          "VERDADEIRO"),
    ("comprovado:",           "VERDADEIRO"),
    ("comprovado -",          "VERDADEIRO"),
    ("sem contexto:",         "FORA_DE_CONTEXTO"),
    ("falta contexto:",       "FORA_DE_CONTEXTO"),
    ("esta fora de contexto", "FORA_DE_CONTEXTO"),
    ("contextualizando:",     "FORA_DE_CONTEXTO"),
]


def _mapear_por_prefixo(chave: str) -> str | None:
    """
    Tenta mapear ratings em forma de sentença a partir do prefixo.
    Retorna None se nenhum prefixo casar.
    """
    for prefixo, categoria in _PREFIXOS_CATEGORIA:
        if chave.startswith(prefixo):
            return categoria
    return None


def mapear_avaliacao(valor: str) -> str:
    """
    Normaliza a avaliação original para uma categoria padronizada.

    Pipeline de resolução (4 etapas):
    1. Lookup direto no mapa (chave normalizada via unicode NFKC)
    2. Lookup com transliteração ASCII (fallback para encoding corrompido)
    3. Fallback por prefixo — chave normalizada (ratings em sentença)
    4. Fallback por prefixo — chave ASCII
    5. OUTRO para tudo que não casar
    """
    if not isinstance(valor, str) or not valor.strip():
        return "NAO_CLASSIFICADO"

    chave       = _normalizar_chave(valor)
    chave_ascii = _ascii_simples(chave)

    return (
        _MAPA_AVALIACAO.get(chave)
        or _MAPA_AVALIACAO.get(chave_ascii)
        or _mapear_por_prefixo(chave)
        or _mapear_por_prefixo(chave_ascii)
        or "OUTRO"
    )


print("Funções utilitárias v2 definidas.")
print(f"  Entradas no mapa de avaliações : {len(_MAPA_AVALIACAO)}")
print(f"  Prefixos de fallback           : {len(_PREFIXOS_CATEGORIA)}")


Funções utilitárias v2 definidas.
  Entradas no mapa de avaliações : 42
  Prefixos de fallback           : 19


## Leitura de todos os CSVs raw

> Arquivos com `_TESTE_` no nome são ignorados automaticamente — eles são gerados em MODO_TESTE e não devem entrar na curadoria oficial.

In [4]:
todos_csvs   = sorted(PASTA_RAW.glob("*.csv"))
arquivos_raw = [a for a in todos_csvs if "_TESTE_" not in a.name]
ignorados    = [a for a in todos_csvs if "_TESTE_" in a.name]

if ignorados:
    print(f"Arquivos _TESTE_ ignorados ({len(ignorados)}):")
    for arq in ignorados:
        print(f"  (ignorado) {arq.name}")

print(f"\nArquivos raw oficiais encontrados: {len(arquivos_raw)}")
for arq in arquivos_raw:
    print(f"  {arq.name}")

if not arquivos_raw:
    raise FileNotFoundError("Nenhum arquivo raw oficial encontrado em " + str(PASTA_RAW))

frames = []
for arq in arquivos_raw:
    df_arq = pd.read_csv(arq, encoding="utf-8-sig", dtype=str)
    df_arq["arquivo_raw_origem"] = arq.name
    frames.append(df_arq)

df_raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal bruto acumulado: {len(df_raw)} registros")
df_raw.head(3)

Arquivos _TESTE_ ignorados (1):
  (ignorado) google_factcheck_raw_TESTE_2026-05-16_21-55-45.csv

Arquivos raw oficiais encontrados: 7
  google_factcheck_raw.csv
  google_factcheck_raw_2026-04-30_00-58-54.csv
  google_factcheck_raw_2026-05-09_15-26-18.csv
  google_factcheck_raw_2026-05-09_15-32-24.csv
  google_factcheck_raw_2026-05-10_02-42-39.csv
  google_factcheck_raw_2026-05-12_23-02-04.csv
  google_factcheck_raw_2026-05-17_02-42-35.csv

Total bruto acumulado: 5726 registros


,termo_busca,texto_afirmacao,data_claim,fonte,url_checagem,avaliacao_original,data_publicacao,arquivo_raw_origem,fonte_verificacao,url_consulta,data_coleta,origem_pipeline
0,urnas eletrônicas,Lula perdeu todas as eleições com votação manu...,2025-12-07T00:00:00Z,AFP Checamos,https://checamos.afp.com/doc.afp.com.88868T3,Enganoso,2025-12-15T17:22:00Z,google_factcheck_raw.csv,NaN,NaN,NaN,NaN
1,urnas eletrônicas,Lula perdeu todas as eleições feitas com cédul...,2025-12-09T00:00:00Z,Aos Fatos,https://www.aosfatos.org/noticias/falso-que-lu...,falso,2025-12-09T00:00:00Z,google_factcheck_raw.csv,NaN,NaN,NaN,NaN
2,urnas eletrônicas,No congresso americano todas as urnas foram ha...,2025-11-02T00:00:00Z,Projeto Comprova,https://projetocomprova.com.br/publica%C3%A7%C...,Falso,2025-11-07T00:00:00Z,google_factcheck_raw.csv,NaN,NaN,NaN,NaN


## Limpeza e padronização

In [5]:
df = df_raw.copy()

# --- texto_principal ---
df["texto_principal"] = df["texto_afirmacao"].apply(remover_html)

# Descartar registros sem texto
antes = len(df)
df = df[df["texto_principal"].str.strip() != ""].copy()
print(f"Removidos sem texto: {antes - len(df)}")

# --- fonte normalizada ---
df["fonte"] = (
    df["fonte_verificacao"]
    .fillna("DESCONHECIDA")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", "_", regex=True)
)

# --- datas padronizadas ---
df["data_publicacao"] = df["data_publicacao"].apply(padronizar_data)

# --- avaliação normalizada ---
df["avaliacao_original"] = df["avaliacao_original"].fillna("").str.strip()
df["avaliacao_categoria"] = df["avaliacao_original"].apply(mapear_avaliacao)

# --- url_origem ---
df["url_origem"] = df["url_checagem"].fillna("").str.strip()

print("Limpeza aplicada.")
print(f"\nDistribuição de avaliacao_categoria:")
print(df["avaliacao_categoria"].value_counts())

Removidos sem texto: 0
Limpeza aplicada.

Distribuição de avaliacao_categoria:
avaliacao_categoria
FALSO               4019
ENGANOSO            1490
FORA_DE_CONTEXTO     131
OUTRO                 50
VERDADEIRO            31
IMPRECISO              5
Name: count, dtype: int64


## Diagnóstico — comparação mapeamento anterior vs. corrigido

Quantifica o ganho da correção executando o mapa **anterior** sobre os mesmos dados.
Executado antes da deduplicação para preservar os números brutos.


In [6]:
# Mapa anterior reproduzido para comparação
_MAPA_ANTERIOR = {
    "falso": "FALSO", "false": "FALSO", "incorreto": "FALSO",
    "incorrect": "FALSO", "mentira": "FALSO",
    "enganoso": "ENGANOSO", "misleading": "ENGANOSO", "distorcido": "ENGANOSO",
    "parcialmente falso": "ENGANOSO", "mostly false": "ENGANOSO", "half true": "ENGANOSO",
    "fora de contexto": "FORA_DE_CONTEXTO", "sem contexto": "FORA_DE_CONTEXTO",
    "out of context": "FORA_DE_CONTEXTO",
    "impreciso": "IMPRECISO", "exagerado": "IMPRECISO",
    "não verificável": "NAO_VERIFICAVEL", "unverified": "NAO_VERIFICAVEL",
    "verdadeiro": "VERDADEIRO", "true": "VERDADEIRO", "comprovado": "VERDADEIRO",
    "certo": "VERDADEIRO", "correto": "VERDADEIRO", "fato": "VERDADEIRO",
    "fato verificado": "VERDADEIRO", "confirmado": "VERDADEIRO",
}

def _mapear_anterior(valor):
    if not isinstance(valor, str) or not valor.strip():
        return "NAO_CLASSIFICADO"
    return _MAPA_ANTERIOR.get(valor.strip().lower(), "OUTRO")

df["_avaliacao_antes"] = df["avaliacao_original"].apply(_mapear_anterior)

antes_dist  = df["_avaliacao_antes"].value_counts()
depois_dist = df["avaliacao_categoria"].value_counts()

sep = "=" * 62
print(sep)
print("  COMPARAÇÃO: MAPEAMENTO ANTERIOR vs. CORRIGIDO  (pré-dedup)")
print(sep)
print()
print(f"  {'Categoria':<24} {'Antes':>8} {'Depois':>8} {'Δ':>8}")
print("  " + "-" * 52)

todas_cats = sorted(set(antes_dist.index) | set(depois_dist.index))
for cat in todas_cats:
    a = antes_dist.get(cat, 0)
    d = depois_dist.get(cat, 0)
    delta = d - a
    sinal = "+" if delta > 0 else ""
    print(f"  {cat:<24} {a:>8} {d:>8} {sinal + str(delta):>8}")

print("  " + "-" * 52)

outro_antes  = antes_dist.get("OUTRO", 0)
outro_depois = depois_dist.get("OUTRO", 0)
recuperados  = outro_antes - outro_depois
print(f"\n  Registros OUTRO recuperados (bruto): {recuperados}")

# Detalhe: destino dos registros que saíram de OUTRO
mask = (df["_avaliacao_antes"] == "OUTRO") & (df["avaliacao_categoria"] != "OUTRO")
recuperados_df = df[mask].copy()

if len(recuperados_df) > 0:
    print("\n  Destino dos registros recuperados:")
    for cat, cnt in recuperados_df["avaliacao_categoria"].value_counts().items():
        print(f"    → {cat:<26} {cnt:>5} registros")

    print("\n  Top avaliacao_original recuperadas de OUTRO:")
    for orig, cnt in recuperados_df["avaliacao_original"].value_counts().head(20).items():
        cat = recuperados_df.loc[recuperados_df["avaliacao_original"] == orig, "avaliacao_categoria"].iloc[0]
        print(f"    {cnt:>4}x  {str(orig)[:60]:<62} → {cat}")

# Valores ainda como OUTRO
ainda_outro = df[df["avaliacao_categoria"] == "OUTRO"]["avaliacao_original"].value_counts()
print(f"\n{sep}")
print(f"  Valores ainda como OUTRO: {ainda_outro.sum()} registros / {len(ainda_outro)} únicos")
print(sep)
for orig, cnt in ainda_outro.head(25).items():
    print(f"  {cnt:>4}x  {str(orig)[:70]}")

print(f"\n  GFC_VERDADEIRO (pré-dedup): {depois_dist.get('VERDADEIRO', 0)}")
print(sep)

# Limpa coluna auxiliar
df.drop(columns=["_avaliacao_antes"], inplace=True)


  COMPARAÇÃO: MAPEAMENTO ANTERIOR vs. CORRIGIDO  (pré-dedup)

  Categoria                   Antes   Depois        Δ
  ----------------------------------------------------
  ENGANOSO                     1222     1490     +268
  FALSO                        3774     4019     +245
  FORA_DE_CONTEXTO              111      131      +20
  IMPRECISO                       5        5        0
  OUTRO                         584       50     -534
  VERDADEIRO                     30       31       +1
  ----------------------------------------------------

  Registros OUTRO recuperados (bruto): 534

  Destino dos registros recuperados:
    → ENGANOSO                     268 registros
    → FALSO                        245 registros
    → FORA_DE_CONTEXTO              20 registros
    → VERDADEIRO                     1 registros

  Top avaliacao_original recuperadas de OUTRO:
     151x  não_é_bem_assim                                                → ENGANOSO
     125x  Errado                      

## Remoção de duplicatas

In [7]:
antes = len(df)
df = df.drop_duplicates(
    subset=["texto_principal", "fonte", "url_origem"],
    keep="first"
).copy()
print(f"Duplicatas removidas: {antes - len(df)}")
print(f"Registros únicos: {len(df)}")

Duplicatas removidas: 1963
Registros únicos: 3763


## Montagem do DataFrame curated

In [8]:
df["id_registro"]      = [str(uuid.uuid4()) for _ in range(len(df))]
df["pipeline"]          = "google_factcheck"
df["tipo_conteudo"]     = "AFIRMACAO_CHECADA"
df["data_curadoria"]    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# rotulo_preliminar reflete a avaliação do fact-checker:
# claims confirmados como verdadeiros recebem VERDADEIRO; demais recebem FALSO.
df["rotulo_preliminar"] = df["avaliacao_categoria"].apply(
    lambda cat: "VERDADEIRO" if cat == "VERDADEIRO" else "FALSO"
)

# Colunas obrigatórias do schema curated
COLUNAS_OBRIGATORIAS = [
    "id_registro",
    "texto_principal",
    "rotulo_preliminar",
    "pipeline",
    "fonte",
    "tipo_conteudo",
    "data_publicacao",
    "url_origem",
    "data_curadoria",
]

# Colunas de contexto específicas deste pipeline
COLUNAS_CONTEXTO = [
    "avaliacao_original",
    "avaliacao_categoria",
    "termo_busca",
    "arquivo_raw_origem",
]

df_curated = df[COLUNAS_OBRIGATORIAS + COLUNAS_CONTEXTO].reset_index(drop=True)

print(f"Shape final do curated: {df_curated.shape}")
print(f"\nColunas: {list(df_curated.columns)}")
df_curated.head(3)

Shape final do curated: (3763, 13)

Colunas: ['id_registro', 'texto_principal', 'rotulo_preliminar', 'pipeline', 'fonte', 'tipo_conteudo', 'data_publicacao', 'url_origem', 'data_curadoria', 'avaliacao_original', 'avaliacao_categoria', 'termo_busca', 'arquivo_raw_origem']


,id_registro,texto_principal,rotulo_preliminar,pipeline,fonte,tipo_conteudo,data_publicacao,url_origem,data_curadoria,avaliacao_original,avaliacao_categoria,termo_busca,arquivo_raw_origem
0,2ce3a9e7-73c6-48dc-94dd-2e28ed41b442,Lula perdeu todas as eleições com votação manu...,FALSO,google_factcheck,DESCONHECIDA,AFIRMACAO_CHECADA,2025-12-15,https://checamos.afp.com/doc.afp.com.88868T3,2026-05-19 00:51:04,Enganoso,ENGANOSO,urnas eletrônicas,google_factcheck_raw.csv
1,e6329802-a1f3-4417-af1c-53899fb27b66,Lula perdeu todas as eleições feitas com cédul...,FALSO,google_factcheck,DESCONHECIDA,AFIRMACAO_CHECADA,2025-12-09,https://www.aosfatos.org/noticias/falso-que-lu...,2026-05-19 00:51:04,falso,FALSO,urnas eletrônicas,google_factcheck_raw.csv
2,e295c494-5a15-4be2-bf62-07a8442bf3ca,No congresso americano todas as urnas foram ha...,FALSO,google_factcheck,DESCONHECIDA,AFIRMACAO_CHECADA,2025-11-07,https://projetocomprova.com.br/publica%C3%A7%C...,2026-05-19 00:51:04,Falso,FALSO,urnas eletrônicas,google_factcheck_raw.csv


## Verificação de qualidade

In [9]:
print("=== Verificação de qualidade ===")
print(f"\nTotal de registros: {len(df_curated)}")
print(f"\nValores nulos por coluna:")
print(df_curated[COLUNAS_OBRIGATORIAS].isnull().sum())
print(f"\nDistribuição por fonte:")
print(df_curated["fonte"].value_counts().head(15))
print(f"\nDistribuição por avaliacao_categoria:")
print(df_curated["avaliacao_categoria"].value_counts())
print(f"\nRegistros com data_publicacao preenchida: {(df_curated['data_publicacao'] != '').sum()}")

=== Verificação de qualidade ===

Total de registros: 3763

Valores nulos por coluna:
id_registro          0
texto_principal      0
rotulo_preliminar    0
pipeline             0
fonte                0
tipo_conteudo        0
data_publicacao      0
url_origem           0
data_curadoria       0
dtype: int64

Distribuição por fonte:
fonte
AOS_FATOS              945
ESTADÃO                754
UOL_NOTÍCIAS           557
AFP_CHECAMOS           533
BOATOS.ORG             291
PROJETO_COMPROVA       230
DESCONHECIDA           191
OBSERVADOR             123
BOL_-_UOL               61
FOLHA_-_UOL             58
METRÓPOLES               7
NEXO_JORNAL              5
AGÊNCIA_TATU             4
CORREIO_BRAZILIENSE      2
BOL                      1
Name: count, dtype: int64

Distribuição por avaliacao_categoria:
avaliacao_categoria
FALSO               2651
ENGANOSO             936
FORA_DE_CONTEXTO     101
OUTRO                 39
VERDADEIRO            31
IMPRECISO              5
Name: count, dtype: int

## Por que esta correção foi necessária

### Contexto

A curadoria anterior usava um mapeamento conservador, projetado para evitar classificações
ambíguas. Isso era correto como postura inicial, mas resultou em ~500 registros classificados
como `OUTRO` que possuíam avaliações claras dos fact-checkers.

### Problemas identificados e corrigidos

**1. Mapa incompleto** — valores comuns sem mapeamento explícito:
- `Errado` (107 ocorrências) → agora `FALSO`
- `Insustentável` (37 ocorrências) → agora `FALSO`
- `Enganador` (15–17 ocorrências) → agora `ENGANOSO`
- `não_é_bem_assim` (91 ocorrências, Boatos.org) → agora `ENGANOSO`
- `Montagem` (9 ocorrências) → agora `FALSO`
- `Sátira` (7 ocorrências) → agora `FALSO`
- `Falta contexto` (4 ocorrências) → agora `FORA_DE_CONTEXTO`
- `confere` (ausente) → agora `VERDADEIRO`

**2. Encoding corrompido** — `não_é_bem_assim` era lido como `n\ufffd\ufffd_\ufffd_bem_assim`
em arquivos raw mais antigos. Corrigido com normalização unicode NFKC + transliteração ASCII.

**3. Ratings em forma de sentença** — fact-checkers como Comprova e AFP Checamos escrevem
o veredicto como frase completa: `"Falso: O vídeo mostra..."`. Nunca casavam com chaves
simples do mapa. Corrigido com fallback por prefixo (`falso:`, `enganoso:`, `comprovado:`, etc.).

### O que foi preservado

- **Nenhum arquivo raw foi alterado** — a camada raw permanece intacta.
- **Arquivos curated anteriores não foram sobrescritos** — novo arquivo tem timestamp próprio.
- **`dataset_final_treino_v1.csv` não foi alterado** — correção afeta apenas a camada curated.

### Valores que continuam como `OUTRO` propositalmente

`Contextualizando`, `Sem registro`, `Explica` e ratings em parágrafo sem prefixo identificável
não têm classificação unívoca e não devem entrar no treino sem revisão manual.


## Exportação para curated/

In [10]:
data_agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
caminho_saida = PASTA_CURATED / f"google_factcheck_curated_{data_agora}.csv"

df_curated.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"Arquivo curated salvo em: {caminho_saida}")
print(f"Total de registros exportados: {len(df_curated)}")
print(f"Data e hora da curadoria: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

Arquivo curated salvo em: ..\dados\pipeline_falso_google_factcheck\curated\google_factcheck_curated_2026-05-19_00-51-04.csv
Total de registros exportados: 3763
Data e hora da curadoria: 19/05/2026 00:51:04


In [11]:
# =====================================================================
# RELATÓRIO FINAL — métricas pós-exportação
# =====================================================================
sep = "=" * 64
print(sep)
print("  RELATÓRIO FINAL — CURADORIA GFC CORRIGIDA (v2)")
print(sep)

dist_final = df_curated["avaliacao_categoria"].value_counts()

print("\n  Distribuição final (pós-dedup):")
print(f"  {'Categoria':<24} {'Registros':>10} {'%':>8}")
print("  " + "-" * 46)
for cat, count in dist_final.items():
    pct = 100 * count / len(df_curated)
    print(f"  {cat:<24} {count:>10}  {pct:>6.1f}%")
print("  " + "-" * 46)
print(f"  {'TOTAL':<24} {len(df_curated):>10}")

n_verdadeiro = dist_final.get("VERDADEIRO", 0)
n_outro      = dist_final.get("OUTRO",      0)

print(f"\n{sep}")
print(f"  GFC_VERDADEIRO (label=1 de alta qualidade): {n_verdadeiro} registros")
print(sep)

verdadeiros_df = df_curated[df_curated["avaliacao_categoria"] == "VERDADEIRO"]
if len(verdadeiros_df) > 0:
    print("\n  Avaliações originais dos GFC_VERDADEIRO:")
    for orig, cnt in verdadeiros_df["avaliacao_original"].value_counts().items():
        print(f"    {cnt:>4}x  {orig}")
    print("\n  Fact-checkers dos GFC_VERDADEIRO:")
    for fonte, cnt in verdadeiros_df["fonte"].value_counts().items():
        print(f"    {cnt:>4}x  {fonte}")

print(f"\n{sep}")
print(f"  OUTRO residual (ambíguos — excluídos do treino): {n_outro} registros")
if n_outro > 0:
    outro_res = df_curated[df_curated["avaliacao_categoria"] == "OUTRO"]["avaliacao_original"].value_counts()
    print("  Principais valores restantes:")
    for orig, cnt in outro_res.head(15).items():
        print(f"    {cnt:>4}x  {str(orig)[:65]}")

print(f"\n{sep}")
print(f"  Arquivo salvo : {caminho_saida}")
print(f"  Data/hora     : {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print(sep)


  RELATÓRIO FINAL — CURADORIA GFC CORRIGIDA (v2)

  Distribuição final (pós-dedup):
  Categoria                 Registros        %
  ----------------------------------------------
  FALSO                          2651    70.4%
  ENGANOSO                        936    24.9%
  FORA_DE_CONTEXTO                101     2.7%
  OUTRO                            39     1.0%
  VERDADEIRO                       31     0.8%
  IMPRECISO                         5     0.1%
  ----------------------------------------------
  TOTAL                          3763

  GFC_VERDADEIRO (label=1 de alta qualidade): 31 registros

  Avaliações originais dos GFC_VERDADEIRO:
      17x  verdadeiro
       8x  Verdadeiro
       3x  Comprovado
       2x  Certo
       1x  COMPROVADO: É verdadeira a comparação de países que usam cloroquina no tratamento da covid-19 e outros que possuem protocolo para uso de cannabis medicinal.

  Fact-checkers dos GFC_VERDADEIRO:
      17x  AOS_FATOS
       7x  UOL_NOTÍCIAS
       2x  PRO